# LangGraph Extraction Pipeline — Walkthrough

This notebook demonstrates the **catalyst-langgraph-aio** orchestration pipeline:
1. Build the extraction graph with a real LLM and mock MCP validators
2. Run the happy path (extract → validate → persist)
3. Run a repair loop (validate rejects → LLM repairs → re-validate)
4. Inspect the state machine and audit trail

> **Kernel**: Run this from the `catalyst-langgraph-aio` venv:  
> `cd libs/catalyst-langgraph-aio && uv run jupyter notebook`

## Setup

In [1]:
import json
import tempfile
from pathlib import Path

from catalyst_langgraph import build_extraction_graph, WorkflowStatus
from catalyst_langgraph.clients import LLMClient, MockMCPClient
from catalyst_langgraph.repository.jsonl import JsonlRepository

# Load environment from project .envrc (works with direnv export format)
from dotenv import load_dotenv

envrc = Path.cwd().parent.parent / ".envrc"
if not envrc.exists():
    envrc = Path.cwd() / ".envrc"  # fallback if running from repo root
if envrc.exists():
    load_dotenv(envrc)
    print(f"Loaded env from {envrc}")
else:
    print("No .envrc found — using shell environment")

Loaded env from /Users/panda/catalyst-devspace/workspace/catalyst-data/.envrc


## Configure LLM

Point at your LiteLLM proxy. The `LLMClient` reads env vars by default.
Set `LLM_API_KEY` in your shell before starting Jupyter.

In [ ]:
import os

# LiteLLM proxy via Traefik ingress — set these in your shell or .env file
os.environ.setdefault("LLM_BASE_URL", "http://litellm.talos00/v1")
os.environ.setdefault("LLM_MODEL", "gpt-4o-mini")
# LLM_API_KEY must be set in your environment; never hardcode secrets in notebooks.
assert os.environ.get("LLM_API_KEY"), "Set LLM_API_KEY in your environment before running this notebook"

llm = LLMClient()

SOURCE_TEXT = (
    "The United Nations was founded in 1945 by 51 countries committed to "
    "maintaining international peace and security. Its headquarters is "
    "located in New York City, United States."
)

# Quick smoke test
response = await llm.complete("Say 'hello' in one word.")
print(f"LLM connected: {response.strip()}")
print(f"Source text: {len(SOURCE_TEXT)} chars")

AssertionError: Set LLM_API_KEY in your environment before running this notebook

## Helper: Build Initial State

The graph operates on an `ExtractionState` dict. Here's the initial state template.

In [ ]:
def make_state(document_id="doc-demo", max_retries=3):
    return {
        "source_metadata": {
            "document_id": document_id,
            "chunk_id": "chunk-001",
            "source": "notebook",
            "domain": "demo",
        },
        "raw_text": SOURCE_TEXT,
        "current_mention_candidates": [],
        "current_proposition_candidates": [],
        "accepted_mentions": [],
        "accepted_propositions": [],
        "latest_mention_validation": {},
        "latest_proposition_validation": {},
        "latest_repair_plan": {},
        "mention_retry_count": 0,
        "proposition_retry_count": 0,
        "max_retries": max_retries,
        "status": "pending",
        "audit_events": [],
        "error": "",
    }

## 1. Happy Path

Both validations pass on the first try. The graph flows:

```
extract_mentions → validate_mentions → extract_propositions → validate_propositions → persist_artifacts
```

In [ ]:
mcp = MockMCPClient({
    "validate_mentions": {"verdict": "valid", "errors": []},
    "validate_propositions": {"verdict": "valid", "errors": []},
})

with tempfile.TemporaryDirectory() as tmp:
    repo = JsonlRepository(Path(tmp))
    graph = build_extraction_graph(llm, mcp, repo)

    result = await graph.ainvoke(make_state("doc-happy"))

    print(f"Status: {result['status']}")
    print(f"Error:  {result.get('error', '')}")
    print(f"Accepted mentions: {len(result['accepted_mentions'])}")
    print(f"Accepted propositions: {len(result['accepted_propositions'])}")
    print(f"Mention retries: {result['mention_retry_count']}")
    print(f"Proposition retries: {result['proposition_retry_count']}")

    # Show audit trail to debug flow
    print(f"\nAudit trail ({len(result['audit_events'])} events):")
    for event in result["audit_events"]:
        details = event.get("details", {})
        detail_str = ", ".join(f"{k}={v}" for k, v in details.items()) if details else ""
        print(f"  {event['node_name']:30s}  {event['status']:12s}  {detail_str}")

    # Check persisted files
    doc_dir = Path(tmp) / "doc-happy"
    if doc_dir.exists():
        for f in sorted(doc_dir.iterdir()):
            lines = f.read_text().strip().split("\n")
            print(f"\n{f.name}: {len(lines)} records")
    else:
        print(f"\nNo files persisted (directory not created)")
        print(f"Temp contents: {list(Path(tmp).iterdir())}")

## 2. Inspect Accepted Mentions

In [ ]:
for m in result["accepted_mentions"]:
    conf = f"  conf={m['confidence']}" if "confidence" in m else ""
    print(f"  {m['surface_form']:20s}  type={m['entity_type']:12s}  span=[{m['start_offset']}:{m['end_offset']}]{conf}")

## 3. Inspect Audit Trail

Every node records an audit event with timestamp, node name, and status.

In [ ]:
for event in result["audit_events"]:
    details = event.get("details", {})
    detail_str = ", ".join(f"{k}={v}" for k, v in details.items()) if details else ""
    print(f"  {event['node_name']:30s}  status={event['status']:12s}  {detail_str}")

## 4. MCP Call Log

The MockMCPClient records every tool call the graph made.

In [ ]:
for tool_name, args in mcp.calls:
    arg_keys = list(args.keys())
    print(f"  {tool_name}({', '.join(arg_keys)})")

## 5. Repair Loop

Now let's simulate the LLM producing bad data on the first try. The MCP validator rejects it, the repair node feeds errors back, and on the second try the LLM produces corrected data.

```
extract_mentions → validate_mentions (REJECTED)
    → repair_mentions → validate_mentions (ACCEPTED)
        → extract_propositions → validate_propositions → persist
```

In [ ]:
# Stateful MCP: reject first call, accept second
call_count = {"n": 0}

def mention_validator(args):
    call_count["n"] += 1
    if call_count["n"] == 1:
        return {
            "verdict": "invalid",
            "errors": ["SPAN_MISMATCH: span [0:14] does not match 'United Nations'"],
        }
    return {"verdict": "valid", "errors": []}

mcp = MockMCPClient()
mcp.set_response("validate_mentions", mention_validator)
mcp.set_response("validate_propositions", {"verdict": "valid", "errors": []})

with tempfile.TemporaryDirectory() as tmp:
    repo = JsonlRepository(Path(tmp))
    graph = build_extraction_graph(llm, mcp, repo)

    result = await graph.ainvoke(make_state("doc-repair"))

    print(f"Status: {result['status']}")
    print(f"Mention retries: {result['mention_retry_count']}  (repaired once!)")
    print(f"Accepted mentions: {len(result['accepted_mentions'])}")
    print(f"\nAudit trail ({len(result['audit_events'])} events):")
    for event in result["audit_events"]:
        print(f"  {event['node_name']:30s}  {event['status']}")

## 6. Max Retries — Graceful Failure

If the LLM keeps producing bad data, the graph gives up after `max_retries` attempts.

In [ ]:
mcp = MockMCPClient({
    "validate_mentions": {"verdict": "invalid", "errors": ["Always fails"]},
})

with tempfile.TemporaryDirectory() as tmp:
    repo = JsonlRepository(Path(tmp))
    graph = build_extraction_graph(llm, mcp, repo)

    result = await graph.ainvoke(make_state("doc-fail", max_retries=2))

    print(f"Status: {result['status']}")
    print(f"Mention retries: {result['mention_retry_count']}  (exhausted max_retries=2)")
    print(f"Last verdict: {result['latest_mention_validation'].get('verdict')}")
    print(f"Propositions extracted: {len(result.get('accepted_propositions', []))}  (never reached)")

    doc_dir = Path(tmp) / "doc-fail"
    print(f"Files persisted: {doc_dir.exists()}  (nothing written)")

## 7. Graph Topology

Here's the actual graph structure rendered as a Mermaid diagram:

```mermaid
graph TD
    START --> extract_mentions
    extract_mentions --> validate_mentions
    validate_mentions -->|valid| extract_propositions
    validate_mentions -->|invalid, retries left| repair_mentions
    validate_mentions -->|max retries| END
    repair_mentions --> validate_mentions
    extract_propositions --> validate_propositions
    validate_propositions -->|valid| persist_artifacts
    validate_propositions -->|invalid, retries left| repair_propositions
    validate_propositions -->|max retries| END
    repair_propositions --> validate_propositions
    persist_artifacts --> END
```

## Summary

| Scenario | Mention Retries | Propositions | Status | Files Written |
|----------|----------------|-------------|--------|---------------|
| Happy path | 0 | 1 | completed | 3 |
| Repair loop | 1 | 1 | completed | 3 |
| Max retries | 2 | 0 | (not completed) | 0 |

---

**Previous**: See [01_mcp_contract_validation.ipynb](./01_mcp_contract_validation.ipynb) for the MCP validation layer this pipeline calls.